![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)


# Introducción a Transformers Architecture


* Profesor : [Carlos Daniel Jiménez Martínez](danieljimenez88m@gmail.com)

* Objetivo de la clase : Presentar como se trabaja computacionalemnte con modelos de lenguaje, de donde provienen y como se crean en si.

## Argumentos Base

* La ingeniería de AI consiste en 
    * Datos : Textos  no estrucutrados y multimodales
    * Procesos donde construir un contexto es lo importante , paso seguido hacer un prompt
    * Adaptación de los modelos (destilación) con fine tuning o eficiencia de los parámetros

* Los modelos core : codifican la información de manera que calcula matemáticamente que sencuiencia tiene mayor probabilidad de seguir en un texto:

    > Fue sin querer 

    > Fue sin querer **queriendo**

* AutoSupervision : Dado que los modelos requieren etiquetas para asi poder entrensarse, la estructura del lenguaje aprende prediciendo partes ocultas en los textyos crudos no etiquetados.

* La ingeniería de AI consiste en usar modelos fundacionales o vía API, donde :

    * Prompting -> Instrucciones
    * Rag -> El Contexto
    * Modelo -> Ajuste internos de los datos para entender como actuar de manera probabilistica

## Bases para que esto funcione de aquí en adelante
### Carácteristicas de un prompt 

Los prompts tanto para agentes como para Chatbots requieren de la siguiente ecuación , no la olvide, usela siempre :


* Role 
* Tarea
* Como debe ser el output
* Ejemplos (Nice to Have)
* Contexto

In [1]:
#######################
# ---- Librerias ---- #
#######################

import os
from openai import OpenAI
from dotenv import load_dotenv
import config  
from IPython.display import display, Markdown
from IPython.display import display, Markdown
client = OpenAI()

In [2]:
from enum import Enum

class OpenAIModels(str, Enum):  # str -> valores como cadenas | Enum -> enumeración
    GPT_5_5 = "gpt-5.5"
    GPT_5_MINI = "gpt-5-mini"

MODEL = OpenAIModels.GPT_5_MINI

In [ ]:
def get_completion(system_prompt,           # Define el comportamiento del modelo
                   user_prompt,             # Solicitud del usuario
                   model=MODEL):

    messages = [{"role": "user", "content": user_prompt}]
    if system_prompt is not None:
        messages = [{"role": "system", "content": system_prompt}, *messages]
    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"An error occurred: {e}"

In [4]:
def display_responses(*args):
    """AYuda visual en formato markdown para comparar los modelos."""
    markdown_string = "<table><tr>"
    for arg in args:
        markdown_string += f"<th>System Prompt:<br />{arg['system_prompt']}<br /><br />"
        markdown_string += f"User Prompt:<br />{arg['user_prompt']}</th>"
    markdown_string += "</tr>"
    markdown_string += "<tr>"
    for arg in args:
        markdown_string += f"<td>Response:<br />{arg['response']}</td>"
    markdown_string += "</tr></table>"
    display(Markdown(markdown_string))

In [5]:
system_prompt = "Eres un experto en los 4 fantasticos, en comics y en historia de la segunda guerra mundial, tus respuestas son laconicas y coherentes y tienen un hilo conductor"
user_prompt = "Escribe un resumen sobre por qué Doom tiene conflicto con los 4 fantasticos"

print('=='*32)
print(f'Enviar solicitud al modelo:  {MODEL}')
baseline_response = get_completion(system_prompt, user_prompt)
print('=='*32)
display_responses({
    "system_prompt": system_prompt,
    "user_prompt": user_prompt, 
    "response": baseline_response
})

Enviar solicitud al modelo:  OpenAIModels.GPT_5_MINI


<table><tr><th>System Prompt:<br />Eres un experto en los 4 fantasticos, en comics y en historia de la segunda guerra mundial, tus respuestas son laconicas y coherentes y tienen un hilo conductor<br /><br />User Prompt:<br />Escribe un resumen sobre por qué Doom tiene conflicto con los 4 fantasticos</th></tr><tr><td>Response:<br />Porque en sus orígenes hay a la vez un agravio personal, una rivalidad intelectual y un choque de proyecto político.

- Origen personal: Victor von Doom y Reed Richards fueron compañeros en la universidad. Un experimento fallido (y el orgullo de Doom al intervenir en prácticas ocultistas y tecnológicas) acabó con la máscara y el rostro desfigurado de Doom; él culpó a Reed por humillación y fracaso.
- Rivalidad intelectual: Doom se considera la mente más brillante y ve a Reed como su único igual —y a la vez su rival—. Esa mezcla de admiración y resentimiento alimenta ataques repetidos.
- Choque de valores: Reed actúa por curiosidad científica y altruismo; Doom gobierna Latveria con mano dura, cree en el orden impuesto y en su derecho a mandar. Para Doom, los fines justifican los medios; para los 4 Fantásticos no.
- Poder y ambición: Doom busca controlar tecnología, magia y, a veces, el tejido mismo de la realidad. Los Fantastic Four, por su papel público y por la amenaza que representan a sus planes, se interponen constantemente.
- Historia y contexto nacional: la experiencia de Doom como hijo de una nación pequeñita y maltratada (Latveria) y las heridas de los conflictos del siglo XX alimentan su voluntad de imponer seguridad y prestigio por la fuerza.
- Ciclo recurrente: hay odio, sí, pero también respeto: Doom frecuentemente declara que Richards es el único capaz de detenerlo y, en ocasiones, se alía con los 4 ante amenazas mayores. Aun así, la combinación de orgullo herido, ansia de poder y diferencias morales hace que el conflicto sea casi eterno.

En resumen: no es solo rencor; es orgullo científico herido más un proyecto político autoritario que choca frontalmente con el idealismo de los 4 Fantásticos.</td></tr></table>

### Transfomers 

* Proceso de tranformación de tokens en paralelo para generar entrenamiento de modelos 

* Esto se basa en modelos atencionales que consuisten en ver la importancia matemática de todos los tokens para generar el siguiente 

* Esto se logra gacvias a Multi-head Attention: sistema masivo de recuperación de información

![](https://upload.wikimedia.org/wikipedia/commons/thumb/3/34/Transformer%2C_full_architecture.png/960px-Transformer%2C_full_architecture.png)



In [6]:
import pandas as pd
import numpy as np
import tiktoken

In [ ]:
text = "Nunca es demasiado tarde para ser sabio."


encoding = tiktoken.get_encoding("o200k_base")

token_ids = encoding.encode(text)
print(f"IDs de Tokens: {token_ids}")

tokens_text = [encoding.decode_single_token_bytes(t).decode("utf-8") for t in token_ids]
print(f"Tokens de texto: {tokens_text}")

df_tokens = pd.DataFrame({"Token ID": token_ids, "Token Text": tokens_text})
print(df_tokens)

IDs de Tokens: [170149, 878, 55219, 26745, 1209, 1334, 6928, 726, 13]
Tokens de texto: ['Nunca', ' es', ' demasiado', ' tarde', ' para', ' ser', ' sab', 'io', '.']
   Token ID  Token Text
0    170149       Nunca
1       878          es
2     55219   demasiado
3     26745       tarde
4      1209        para
5      1334         ser
6      6928         sab
7       726          io
8        13           .


In [8]:
response = client.embeddings.create(
    input=text,
    model="text-embedding-3-small"
)

embedding_vector = response.data[0].embedding
print(f"Dimensión del vector: {len(embedding_vector)}") 
print(f"Primeros 15 valores: {embedding_vector[:15]}")

Dimensión del vector: 1536
Primeros 15 valores: [0.049468994140625, 0.00640106201171875, -0.0230560302734375, 0.0285797119140625, 0.032318115234375, 0.005340576171875, -0.0027008056640625, 0.037109375, -0.03973388671875, 0.006008148193359375, 0.0162811279296875, -0.006717681884765625, 0.0200347900390625, -0.017974853515625, 0.0269317626953125]


In [10]:
prompt = "La mejor canción de Red Hot Chilli papers es "


response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=15,
    logprobs=True,      # habilita la inspección de probabilidades
    top_logprobs=5,     # top-5 alternativas por token
)

generated_content = response.choices[0].message.content
print(f"Prompt: {prompt}")
print(f"Completado: {generated_content}\n")

for i, tok in enumerate(response.choices[0].logprobs.content):
    print(f"Token {i}: {tok.token!r}  logprob={tok.logprob:.4f}")
    for alt in tok.top_logprobs:
        print(f"    alt: {alt.token!r}  logprob={alt.logprob:.4f}")

Prompt: La mejor canción de Red Hot Chilli papers es 
Completado: La mejor canción de Red Hot Chili Peppers es un tema muy subjetivo

Token 0: 'La'  logprob=-0.3562
    alt: 'La'  logprob=-0.3562
    alt: 'Eleg'  logprob=-1.2312
    alt: 'Determ'  logprob=-5.6062
    alt: 'Dec'  logprob=-6.3562
    alt: 'Es'  logprob=-7.3562
Token 1: ' mejor'  logprob=-0.6851
    alt: ' mejor'  logprob=-0.6851
    alt: ' elección'  logprob=-0.9351
    alt: ' "'  logprob=-2.6851
    alt: ' opinión'  logprob=-3.5601
    alt: ' respuesta'  logprob=-5.3101
Token 2: ' canción'  logprob=0.0000
    alt: ' canción'  logprob=0.0000
    alt: ' canciones'  logprob=-17.7500
    alt: ' Canc'  logprob=-19.1250
    alt: ' song'  logprob=-19.2500
    alt: ' can'  logprob=-20.2500
Token 3: ' de'  logprob=-0.0000
    alt: ' de'  logprob=-0.0000
    alt: ' del'  logprob=-16.6250
    alt: ' es'  logprob=-19.8750
    alt: ' puede'  logprob=-20.5000
    alt: ' depende'  logprob=-21.6250
Token 4: ' Red'  logprob=-0.0486
    

In [11]:
print("--- Inspección de Probabilidades (Looking Inside) ---")
for i, token_data in enumerate(response.choices[0].logprobs.content):
    token = token_data.token
    prob = np.exp(token_data.logprob) * 100  # Convertir logprob a % real
    
    print(f"\nPaso {i+1}: Token elegido -> '{token}' ({prob:.2f}% confianza)")
    
    print("   Otras opciones consideradas:")
    for top_k in token_data.top_logprobs:
        top_prob = np.exp(top_k.logprob) * 100
        print(f"     - '{top_k.token}': {top_prob:.2f}%")

--- Inspección de Probabilidades (Looking Inside) ---

Paso 1: Token elegido -> 'La' (70.03% confianza)
   Otras opciones consideradas:
     - 'La': 70.03%
     - 'Eleg': 29.19%
     - 'Determ': 0.37%
     - 'Dec': 0.17%
     - 'Es': 0.06%

Paso 2: Token elegido -> ' mejor' (50.40% confianza)
   Otras opciones consideradas:
     - ' mejor': 50.40%
     - ' elección': 39.26%
     - ' "': 6.82%
     - ' opinión': 2.84%
     - ' respuesta': 0.49%

Paso 3: Token elegido -> ' canción' (100.00% confianza)
   Otras opciones consideradas:
     - ' canción': 100.00%
     - ' canciones': 0.00%
     - ' Canc': 0.00%
     - ' song': 0.00%
     - ' can': 0.00%

Paso 4: Token elegido -> ' de' (100.00% confianza)
   Otras opciones consideradas:
     - ' de': 100.00%
     - ' del': 0.00%
     - ' es': 0.00%
     - ' puede': 0.00%
     - ' depende': 0.00%

Paso 5: Token elegido -> ' Red' (95.25% confianza)
   Otras opciones consideradas:
     - ' Red': 95.25%
     - ' los': 4.74%
     - ' la': 0.00%
  